# Final French evaluation on Kaggle GPU

Enable **T4 x2** and **Internet**. Add the private `constitution-evidence-final-input.zip` bundle as a Kaggle input. This recovery bundle preserves the 90 BM25 and 90 hybrid rows and runs only the 90 full-text questions. The pinned Qwen3-4B model is split across both GPUs; its long prompt is prefetched in 512-token chunks into GPU memory. Run the cells in order. The first install may require a kernel restart; then run all cells again. The GPU evaluation runs in a separate process so its memory is released if the run stops. Keep the input dataset and downloaded results private.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, time, zipfile
INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/moroccan-constitution-rag')
archives = sorted(INPUT.rglob('constitution-evidence-final-input.zip'))
folders = sorted({p.parent.parent for p in INPUT.rglob('kaggle-bundle.json') if p.parent.name == 'runs'})
if len(archives) > 1 or (not archives and len(folders) != 1):
    raise RuntimeError(f'Expected one private final input; archives={archives}, extracted folders={folders}')
if archives:
    with zipfile.ZipFile(archives[0]) as bundle:
        incoming = json.loads(bundle.read('runs/kaggle-bundle.json'))
else:
    incoming = json.loads((folders[0]/'runs/kaggle-bundle.json').read_text())
if ROOT.exists():
    existing_path = ROOT/'runs/kaggle-bundle.json'
    existing = json.loads(existing_path.read_text()) if existing_path.exists() else {}
    if existing.get('commit') != incoming['commit']:
        previous = ROOT/'runs/final-reserved-qwen4b-transformers'
        if previous.exists():
            files = [path for path in previous.rglob('*') if path.is_file()]
            release = ROOT/'runs/release-candidate/manifest.json'
            if release.is_file():
                files.append(release)
            if files:
                backup = ROOT.parent/f'constitution-final-results-before-upgrade-{time.time_ns()}.zip'
                with zipfile.ZipFile(backup, 'w', compression=zipfile.ZIP_DEFLATED) as saved:
                    for path in files:
                        saved.write(path, path.relative_to(ROOT).as_posix())
                with zipfile.ZipFile(backup) as saved:
                    if saved.testzip() is not None:
                        raise RuntimeError(f'Could not verify the backup ZIP: {backup}')
                print('Saved previous results:', backup)
            fulltext = previous/'fulltext/answers.jsonl'
            successful = 0
            if fulltext.exists():
                with fulltext.open() as rows:
                    successful = sum(
                        1 for line in rows if line.strip()
                        and (row := json.loads(line)).get('error') is None and row.get('answer'))
            if successful:
                raise RuntimeError(
                    f'Previous run has {successful} successful full-text answers. '
                    f'They were backed up at {backup}; use that ZIP for a recovery bundle before upgrading.')
        shutil.rmtree(ROOT)
if archives:
    with zipfile.ZipFile(archives[0]) as bundle:
        for name in bundle.namelist():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts:
                raise RuntimeError('Unsafe archive path')
        ROOT.mkdir(parents=True, exist_ok=True)
        bundle.extractall(ROOT)
else:
    shutil.copytree(folders[0], ROOT, dirs_exist_ok=True)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'src'))
from scripts.kaggle_bundle_loader import restore_git_metadata
print('Bundle commit:', restore_git_metadata(ROOT))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())


In [ ]:
import hashlib, os
requirements = ['-e','.[dense]','transformers==4.57.6','accelerate>=1,<2',
                'bitsandbytes>=0.46,<1','huggingface_hub>=0.35,<1']
signature = hashlib.sha256(json.dumps(requirements).encode()).hexdigest()
marker = Path('/kaggle/working/constitution-final-env.txt')
if not marker.exists() or marker.read_text().strip() != signature:
    subprocess.run([sys.executable,'-m','pip','install','-q',*requirements],check=True,cwd=ROOT)
    marker.write_text(signature+'\n')
    raise RuntimeError('Packages installed. Restart the Kaggle kernel, then rerun all cells.')
print('Dependencies installed; the GPU check runs in the isolated evaluation process.')


In [ ]:
from scripts.kaggle_final_runtime import verify_bundle
from constitution_evidence.release import release_readiness
result = verify_bundle(ROOT)
readiness = release_readiness(ROOT/'runs/complete-owner-reviewed-benchmark.jsonl',
    ROOT/'data/processed/manifest.json', ROOT/'runs/validation-selection.json',
    ROOT/'configs/final_systems.json', repo_root=ROOT)
if readiness['status'] != 'ready':
    raise RuntimeError(f"Release gate blocked: {readiness['reasons']}")
print('Ready at commit',result['commit'],'with',len(result['files']),'verified private inputs')


In [ ]:
# The worker saves each answer row immediately. After an error, run the export cell below.
subprocess.run([sys.executable, '-m', 'scripts.kaggle_final_runtime', str(ROOT)],
               check=True, cwd=ROOT)


In [ ]:
from scripts.kaggle_final_runtime import export_results
artifact = export_results(ROOT,Path('/kaggle/working/constitution-final-results.zip'))
print(json.dumps(artifact,ensure_ascii=False,indent=2))
print('Download this ZIP from the Kaggle Output panel before the session ends.')


The result ZIP includes recovery lineage, the reused BM25 and hybrid rows, and new full-text rows. The evaluation worker runs in a separate process and stops after the first CUDA or execution error. If a session ends, export the partial ZIP; successful full-text rows can be reused in the next private bundle. This is a documented post hoc infrastructure repair, not a new blind test.
